# TinyYOLO Experiment Suite — Notebook 1/7
## VOC 2007+2012 Benchmark (Table I & V)

**What:** Train both variants (quantized + standard) on VOC with 5 seeds each for statistical rigor.

**GPU Time:** ~52h T4 (10 runs × ~5.2h each)

**Platform Notes:**
- **RunPod/Vast.ai:** Run all cells — no session limits.
- **Kaggle (12h):** Run ONE variant per session (set `RUN_VARIANTS` below).
- **Colab Free (12h):** Same as Kaggle — one variant per session.
- **Colab Pro / Local GPU:** Run all cells.

**Results → `experiments/results/voc-{q,s}-416-seed*/summary.json`**

In [ ]:
#@title ⚙️ Configuration — Edit this cell

# Which variants to run in THIS session?
# Options: ['quantized', 'standard']  or  ['quantized']  or  ['standard']
RUN_VARIANTS = ['quantized', 'standard']  #@param

# Which seeds to run? Full: [42, 123, 256, 512, 1024]
SEEDS = [42, 123, 256, 512, 1024]  #@param

EPOCHS = 300
IMGSZ = 416
BATCH = 128  # Auto-reduced if OOM
REPO = 'https://github.com/ShMazumder/tinyYOLO.git'

In [ ]:
#@title 📦 Environment Setup
import os, sys, socket, subprocess, shutil
from pathlib import Path

# Platform detection
platform = 'unknown'
if os.path.exists('/kaggle'):
    platform = 'kaggle'
elif 'COLAB_GPU' in os.environ or 'COLAB_RELEASE_TAG' in os.environ:
    platform = 'colab'
elif os.path.exists('/workspace'):
    platform = 'runpod'
elif os.environ.get('VAST_CONTAINERLABEL'):
    platform = 'vast'
else:
    platform = 'local'
print(f'🖥️  Platform: {platform}')

# Internet check
try:
    socket.create_connection(('github.com', 80), timeout=5)
    print('🌐 Internet: OK')
except:
    print('❌ No internet! On Kaggle, toggle Internet ON in sidebar.')
    sys.exit(1)

# Clone repo
if Path('tinyYOLO').exists():
    shutil.rmtree('tinyYOLO')
!git clone {REPO} --depth 1
%cd tinyYOLO

# Install
!pip install -e . -q 2>&1 | tail -1
!pip install tqdm timm psutil -q 2>&1 | tail -1

# GPU check
import torch
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f'🔥 GPU: {gpu} ({mem:.1f} GB)')
else:
    print('⚠️  No GPU detected — training will be very slow!')

# Download VOC
print('\n📁 Downloading VOC dataset...')
!python3 -c "from ultralytics.data.utils import check_det_dataset; check_det_dataset('VOC.yaml')" 2>&1 | tail -3
print('✅ Setup complete!')

In [ ]:
#@title 🚀 Run VOC Training
import time

total_runs = len(RUN_VARIANTS) * len(SEEDS)
run_i = 0

for variant in RUN_VARIANTS:
    tag = 'q' if variant == 'quantized' else 's'
    for seed in SEEDS:
        run_i += 1
        name = f'voc-{tag}-{IMGSZ}-seed{seed}'
        result_dir = Path(f'experiments/results/{name}')

        # Skip if already completed
        if (result_dir / 'summary.json').exists():
            print(f'⏭️  [{run_i}/{total_runs}] {name} — already done, skipping')
            continue

        print(f'\n{"="*60}')
        print(f'🏃 [{run_i}/{total_runs}] {name}')
        print(f'{"="*60}')
        t0 = time.time()

        !python3 scripts/train.py --task det --variant {variant} --lr 2e-4 --no-amp --freeze-epochs 5 --grad-clip 5.0 --val-conf 0.001 --ema-decay 0.9998 --close-mosaic 150 \
            --data voc.yaml --imgsz {IMGSZ} --epochs {EPOCHS} \
            --batch {BATCH} --seed {seed} --warmup 3 \
            --pretrained --compile --name {name}

        elapsed = (time.time() - t0) / 3600
        print(f'⏱️  {name} completed in {elapsed:.1f}h')

print(f'\n✅ VOC benchmark complete! {run_i} runs finished.')

In [ ]:
#@title 📊 Collect Results
import json, glob
import numpy as np

print('=' * 70)
print('VOC BENCHMARK RESULTS')
print('=' * 70)

for variant, tag in [('quantized', 'q'), ('standard', 's')]:
    maps = []
    for d in sorted(glob.glob(f'experiments/results/voc-{tag}-{IMGSZ}-seed*')):
        s = Path(d) / 'summary.json'
        if s.exists():
            with open(s) as f:
                data = json.load(f)
            m = data.get('best_map50', 0) * 100
            maps.append(m)
            print(f'  {Path(d).name}: {m:.1f}%')

    if maps:
        print(f'  → TinyYOLO-{tag} mean: {np.mean(maps):.1f} ± {np.std(maps):.1f}%')
    print()

# Save consolidated
out = {'experiment': 'voc_benchmark', 'imgsz': IMGSZ, 'epochs': EPOCHS}
for tag in ['q', 's']:
    maps = []
    for d in sorted(glob.glob(f'experiments/results/voc-{tag}-{IMGSZ}-seed*')):
        s = Path(d) / 'summary.json'
        if s.exists():
            with open(s) as f: data = json.load(f)
            maps.append(data.get('best_map50', 0) * 100)
    out[f'tinyyolo_{tag}'] = {'maps': maps, 'mean': float(np.mean(maps)), 'std': float(np.std(maps))} if maps else {}

with open('experiments/results/voc_benchmark_summary.json', 'w') as f:
    json.dump(out, f, indent=2)
print('Saved: experiments/results/voc_benchmark_summary.json')